# EDR AI Agent - Phân Tích Khám Phá Đặc Trưng (Exploratory Data Analysis - EDA)

Notebook này thực hiện Phân tích Khám phá Dữ liệu (EDA) trên vector đặc trưng 25 chiều được trích xuất bởi EDR Agent từ các sự kiện hệ thống. Mục tiêu của phân tích bao gồm:
1. **Phân tích phân phối lớp**: Đánh giá số lượng mẫu và độ cân bằng giữa các nhãn dữ liệu (Benign - Lành tính, Malware - Mã độc, Credential Access - Chiếm quyền truy cập).
2. **Xác định đặc trưng cốt lõi**: Phân tích sự khác biệt về giá trị trung bình giữa các lớp dữ liệu để tìm ra các đặc trưng có tính phân loại cao.
3. **Ma trận tương quan**: Trực quan hóa tương quan giữa các đặc trưng (sử dụng Heatmap đầy đủ).
4. **Độ quan trọng đặc trưng (Feature Importance)**: Sử dụng thuật toán Random Forest Classifier để chấm điểm độ quan trọng của từng đặc trưng.

In [ ]:
# Tải các thư viện Python cần thiết
import os
import json
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier

# Thiết lập cấu hình trực quan hóa
sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
# Khai báo 25 tên đặc trưng theo thứ tự chỉ số (index) từ file features.h
FEATURE_NAMES = [
    "child_spawn_count_5s",
    "child_spawn_count_30s",
    "child_spawn_count_300s",
    "cmdline_length",
    "cmdline_entropy",
    "has_encoded_cmdline",
    "has_download_cradle",
    "cmdline_suspicious_kw_count",
    "is_lolbin",
    "parent_is_lolbin",
    "token_elevated",
    "process_depth_in_tree",
    "parent_is_office",
    "parent_is_browser",
    "parent_is_script_engine",
    "is_in_temp_path",
    "is_in_system_path",
    "lifetime_ms_log",
    "unusual_parent_child",
    "process_rarity_score",
    "tree_fan_out_max",
    "lsass_access",
    "access_rights_vm_read",
    "persistence_key_access",
    "reg_sam_security_access"
]

def load_features(db_path, label):
    """
    Tải dữ liệu từ database SQLite của từng loại nhãn sự kiện và định dạng thành DataFrame.
    """
    if not os.path.exists(db_path):
        print(f"[Cảnh báo] File {db_path} không tồn tại.")
        return pd.DataFrame()
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    try:
        cursor.execute("SELECT features FROM telemetry_events;")
        rows = cursor.fetchall()
    except sqlite3.OperationalError as e:
        print(f"[Lỗi] Không tìm thấy bảng telemetry_events trong {db_path}: {e}")
        conn.close()
        return pd.DataFrame()
        
    records = []
    for row in rows:
        feats_str = row[0]
        if not feats_str:
            continue
        try:
            feats = json.loads(feats_str)
            if len(feats) == 25:
                records.append(feats)
        except Exception:
            continue
            
    conn.close()
    df = pd.DataFrame(records, columns=FEATURE_NAMES)
    df["label"] = label
    return df

In [ ]:
# Thực hiện tải các tập dữ liệu
data_dir = "data/raw"
df_benign = load_features(os.path.join(data_dir, "benign_events.db"), label=0)
df_malware = load_features(os.path.join(data_dir, "malware_events.db"), label=1)
df_cred = load_features(os.path.join(data_dir, "credential_events.db"), label=2)

# Ghép các bảng dữ liệu lại với nhau
df = pd.concat([df_benign, df_malware, df_cred], ignore_index=True)
print(f"Tổng số mẫu đã tải: {len(df)}")
print(f" - Số mẫu Lành tính (Benign - Label 0): {len(df_benign)}")
print(f" - Số mẫu Mã độc (Malware - Label 1): {len(df_malware)}")
print(f" - Số mẫu Chiếm quyền (Credential Access - Label 2): {len(df_cred)}")

In [ ]:
# Ánh xạ tên lớp sự kiện để phục vụ hiển thị biểu đồ
class_map = {0: "Benign", 1: "Malware", 2: "Credential Access"}
df["class_name"] = df["label"].map(class_map)
df.head()

## 1. Giá trị trung bình các đặc trưng theo lớp dữ liệu

Chúng ta sẽ tính toán giá trị trung bình cho tất cả 25 đặc trưng của từng lớp dữ liệu để xem đặc trưng nào có độ biến thiên cao nhất giữa các lớp lành tính và độc hại.

In [ ]:
# Tính toán giá trị trung bình
means_df = df.groupby("class_name")[FEATURE_NAMES].mean().T
means_df

## 2. Trực quan hóa các đặc trưng trọng yếu

### A. Chỉ thị hành vi chiếm quyền truy cập (Credential Access)
Chúng ta sẽ vẽ biểu đồ cột cho `lsass_access` và `access_rights_vm_read` để xem sự phân bố đối với tiến trình dump bộ nhớ LSASS.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=df, x="class_name", y="lsass_access", ax=axes[0], hue="class_name", legend=False, palette="viridis")
axes[0].set_title("Giá trị trung bình lsass_access theo nhãn")
axes[0].set_ylabel("Giá trị trung bình (0.0 - 1.0)")

sns.barplot(data=df, x="class_name", y="access_rights_vm_read", ax=axes[1], hue="class_name", legend=False, palette="viridis")
axes[1].set_title("Giá trị trung bình access_rights_vm_read theo nhãn")
axes[1].set_ylabel("Giá trị trung bình (0.0 - 1.0)")
plt.tight_layout()
plt.show()

### B. Chỉ thị hành vi mã độc (Malware Indicators)
Trực quan hóa cờ lệnh dòng lệnh mã hóa (`has_encoded_cmdline`), số từ khóa nghi vấn (`cmdline_suspicious_kw_count`), và cờ lệnh tải file từ môi trường mạng (`has_download_cradle`).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(data=df, x="class_name", y="has_encoded_cmdline", ax=axes[0], hue="class_name", legend=False, palette="magma")
axes[0].set_title("Tỷ lệ dòng lệnh mã hóa (-enc / -encoded)")

sns.barplot(data=df, x="class_name", y="cmdline_suspicious_kw_count", ax=axes[1], hue="class_name", legend=False, palette="magma")
axes[1].set_title("Số lượng từ khóa dòng lệnh nghi vấn")

sns.barplot(data=df, x="class_name", y="has_download_cradle", ax=axes[2], hue="class_name", legend=False, palette="magma")
axes[2].set_title("Tỷ lệ lệnh tải file trực tuyến")
plt.tight_layout()
plt.show()

### C. Vị trí chạy file và Điểm độ hiếm tiến trình
Khảo sát xem tiến trình có chạy từ thư mục tạm (`is_in_temp_path`) và điểm độ hiếm của tiến trình hệ thống (`process_rarity_score`) biến thiên thế nào.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=df, x="class_name", y="is_in_temp_path", ax=axes[0], hue="class_name", legend=False, palette="coolwarm")
axes[0].set_title("Tỷ lệ tiến trình chạy từ Temp Path")

sns.barplot(data=df, x="class_name", y="process_rarity_score", ax=axes[1], hue="class_name", legend=False, palette="coolwarm")
axes[1].set_title("Độ hiếm tiến trình (Process Rarity)")
plt.tight_layout()
plt.show()

## 3. Phân tích ma trận tương quan (Feature Correlation)

**Lưu ý quan trọng về biểu đồ Heatmap của `child_spawn_count_5s` (Index 0)**:
* Nếu biểu đồ sử dụng mặt nạ (mask) tam giác dưới che đi đường chéo chính, hàng đầu tiên đại diện cho đặc trưng thứ 0 (`child_spawn_count_5s`) sẽ trông như không có dữ liệu vì tất cả ô trên hàng đó bị che. 
* Để sửa vấn đề này, chúng ta sẽ vẽ ma trận tương quan **đầy đủ** (không che), hiển thị rõ ràng mọi giá trị tương quan của `child_spawn_count_5s` tại cả hàng đầu tiên và cột đầu tiên.
* Ngoài ra, đặc trưng `persistence_key_access` (Index 23) có giá trị không thay đổi (luôn là 0) trên tập mẫu này, dẫn tới độ lệch chuẩn bằng 0 và giá trị tương quan bằng `NaN` (hiển thị dưới dạng ô trắng trống).

In [ ]:
plt.figure(figsize=(16, 14))
corr = df[FEATURE_NAMES].corr()

# Vẽ heatmap đầy đủ, không áp dụng mặt nạ để đảm bảo hiển thị đủ 25 đặc trưng
sns.heatmap(corr, cmap="coolwarm", annot=False, fmt=".2f", linewidths=.5)
plt.title("Ma trận tương quan đặc trưng (Feature Correlation Matrix - Full)")
plt.show()

## 4. Xếp hạng tầm quan trọng đặc trưng (Feature Importance)

Chúng ta huấn luyện mô hình Random Forest Classifier trên toàn bộ tập dữ liệu mẫu để chấm điểm tầm quan trọng Gini của từng đặc trưng.

In [ ]:
X = df[FEATURE_NAMES].values
y = df["label"].values

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

# Sắp xếp chỉ số độ quan trọng giảm dần
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

imp_df = pd.DataFrame({
    "Feature": [FEATURE_NAMES[i] for i in indices],
    "Importance": importances[indices]
})

plt.figure(figsize=(12, 8))
sns.barplot(data=imp_df, x="Importance", y="Feature", hue="Feature", legend=False, palette="viridis")
plt.title("Xếp hạng độ quan trọng đặc trưng (Random Forest Classifier)")
plt.xlabel("Gini Importance")
plt.ylabel("Đặc trưng")
plt.tight_layout()
plt.show()

## Tổng hợp kết quả chính từ EDA

1. **Chỉ thị hành vi chiếm đoạt thông tin**: `lsass_access` (Index 21) và `access_rights_vm_read` (Index 22) cực kỳ thưa thớt nhưng lại là đặc trưng tuyệt đối phân loại lớp Chiếm quyền truy cập (Credential Access).
2. **Vai trò của Tiến trình cha độc hại**: `parent_is_script_engine` (tiến trình cha là powershell, cmd, wscript...) và `parent_is_lolbin` là hai đặc trưng hàng đầu giúp phát hiện mã độc đang chạy lan truyền.
3. **Tương quan giữa các nhóm đặc trưng**: Các đặc trưng liên quan đến thời gian đo lường như `child_spawn_count_5s`, `child_spawn_count_30s`, và `child_spawn_count_300s` có độ tương quan dương mạnh mẽ với nhau (r > 0.85), chứng tỏ chúng mô tả cùng một kiểu hành vi bùng nổ tiến trình con ở các thang đo thời gian khác nhau.